# 7) M1.2 Your first PyPSA network — Exercises

**Goals:** read PyPSA's static and time-varying data, compute available energy and curtailment, run scenarios and measure their effect, handle an infeasible system with load shedding, and swap in real Croatian load.

The setup cell rebuilds and optimises the lecture network (`n`) from the same synthetic profiles. Tasks 1–6 use it; task 7 needs your ENTSO-E token.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import math
import os
import warnings

import numpy as np
import pandas as pd
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)


def synthetic_profiles(start: str = "2025-01-13", days: int = 7, seed: int = 1) -> pd.DataFrame:
    """SYNTHETIC hourly profiles (UTC): load in MW, wind and solar as per-unit availability.

    Shapes are plausible for a Croatia-sized winter week but are NOT measured data.
    Real load comes from ENTSO-E (M1.1), real wind/solar profiles in M1.4.
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=days * 24, freq="h")          # naive timestamps = UTC
    local = idx.tz_localize("UTC").tz_convert("Europe/Zagreb")
    hour, weekend = local.hour.to_numpy(), (local.dayofweek >= 5)
    load = (2150 + 380 * np.sin((hour - 7) / 24 * 2 * np.pi) - 200 * weekend
            + rng.normal(0, 40, len(idx)))
    wind = np.empty(len(idx))
    wind[0] = 0.35
    for t in range(1, len(idx)):                                        # slowly varying wind
        wind[t] = np.clip(0.9 * wind[t - 1] + 0.1 * 0.35 + rng.normal(0, 0.06), 0.02, 0.95)
    solar = np.clip(np.sin(np.pi * (hour - 7.5) / 9), 0, None) * rng.uniform(0.3, 0.8, len(idx))
    return pd.DataFrame({"load_MW": load.round(1), "wind_pu": wind.round(3),
                         "solar_pu": solar.round(3)}, index=idx)


def build_network(profiles: pd.DataFrame, with_imports: bool = True) -> pypsa.Network:
    """Stylised single-bus Croatia-like system. Round numbers for teaching, not official data."""
    n = pypsa.Network()
    n.set_snapshots(profiles.index)
    n.add("Carrier", "AC")
    n.add("Bus", "HR", carrier="AC")

    # carriers: co2_emissions in t CO2 per MWh of fuel (thermal)
    for name, co2 in [("nuclear", 0.0), ("hydro", 0.0), ("wind", 0.0), ("solar", 0.0),
                      ("coal", 0.34), ("gas", 0.20), ("imports", 0.0)]:
        n.add("Carrier", name, co2_emissions=co2)

    n.add("Load", "demand", bus="HR", carrier="AC", p_set=profiles["load_MW"])
    #     name           carrier    p_nom MW  marginal cost EUR/MWh  efficiency
    plants = [("nuclear_share", "nuclear", 350, 10, 0.33),
              ("hydro_ror", "hydro", 1500, 2, 1.0),
              ("wind", "wind", 1000, 0, 1.0),
              ("solar", "solar", 800, 0, 1.0),
              ("coal", "coal", 210, 95, 0.38),
              ("gas_ccgt", "gas", 700, 110, 0.58),
              ("gas_ocgt", "gas", 300, 160, 0.38)]
    for name, carrier, p_nom, mc, eff in plants:
        n.add("Generator", name, bus="HR", carrier=carrier, p_nom=p_nom,
              marginal_cost=mc, efficiency=eff)
    n.generators_t.p_max_pu["hydro_ror"] = 0.45      # run-of-river, flat (simplification)
    n.generators_t.p_max_pu["wind"] = profiles["wind_pu"]
    n.generators_t.p_max_pu["solar"] = profiles["solar_pu"]
    if with_imports:
        n.add("Generator", "imports", bus="HR", carrier="imports", p_nom=1500, marginal_cost=120)
    return n


profiles = synthetic_profiles()
n = build_network(profiles)
n.optimize(solver_name="highs", log_to_console=False)

### Warm-ups

**1. Capacity by carrier**

Return the installed capacity (MW) per carrier as a Series, using PyPSA's static generator table.

In [ ]:
def capacity_by_carrier(n: pypsa.Network) -> pd.Series:
    """Installed generator capacity (MW) per carrier."""
    ...  # YOUR CODE HERE

In [ ]:
cap = capacity_by_carrier(n)
assert cap["gas"] == 1000, "two gas plants: 700 + 300 MW"
assert math.isclose(cap.sum(), n.generators.p_nom.sum())
cap

**2. Available energy**

Return the energy (MWh) each generator *could* have produced over all snapshots: capacity × availability, summed over hours. Remember that generators without a time series are fully available (hint: `get_switchable_as_dense`).

In [ ]:
def available_energy_MWh(n: pypsa.Network) -> pd.Series:
    """Maximum producible energy (MWh) per generator over all snapshots (hourly snapshots)."""
    ...  # YOUR CODE HERE

In [ ]:
avail = available_energy_MWh(n)
assert math.isclose(avail["wind"], 1000 * profiles["wind_pu"].sum()), "wind: p_nom x p_max_pu"
assert math.isclose(avail["coal"], 210 * len(n.snapshots)), "no time series -> available every hour"
assert (n.generators_t.p.sum() <= avail + 1e-6).all(), "nobody can produce more than available"

### Core

**3. Curtailment**

**Curtailment** is renewable energy that was available but not used. Return curtailed energy (MWh) for the variable renewables (`wind`, `solar`, `hydro_ror`) of a solved network. Then check it in a system with far more solar.

In [ ]:
def curtailment_MWh(n: pypsa.Network, generators=("wind", "solar", "hydro_ror")) -> pd.Series:
    """Available minus dispatched energy (MWh) for the given generators of a solved network."""
    ...  # YOUR CODE HERE

In [ ]:
assert curtailment_MWh(n).sum() < 1e-3, "in the base case every renewable MWh is used"

big_solar = build_network(profiles)
big_solar.generators.loc["solar", "p_nom"] = 4000
big_solar.optimize(solver_name="highs", log_to_console=False)
curt = curtailment_MWh(big_solar)
assert curt.sum() > 1000, "with 4 GW of solar, midday supply exceeds demand"
assert (curt >= 0).all()
curt.round(0)

**4. Scenario runner**

Write `run_scenario`: build the network, set the solar capacity, optimise, and return a dict with `total_cost_EUR`, `emissions_t` (use `co2_emissions / efficiency`, as in the lecture), `avg_price_EUR_per_MWh` (simple mean over hours) and `hours_at_import_price` (hours where the price is 120 €/MWh). Compare 800 MW and 1,600 MW of solar.

In [ ]:
def run_scenario(profiles: pd.DataFrame, solar_MW: float) -> dict:
    """Optimise the stylised system with a given solar capacity and summarise the outcome."""
    ...  # YOUR CODE HERE

In [ ]:
base, more_solar = run_scenario(profiles, 800), run_scenario(profiles, 1600)
assert math.isclose(base["total_cost_EUR"], n.objective), "800 MW is the lecture system"
assert more_solar["total_cost_EUR"] < base["total_cost_EUR"]
assert more_solar["emissions_t"] <= base["emissions_t"]
assert more_solar["avg_price_EUR_per_MWh"] <= base["avg_price_EUR_per_MWh"]
pd.DataFrame({"800 MW solar": base, "1600 MW solar": more_solar}).round(1)

**5. Interpretation**

Doubling solar lowered cost, emissions and the average price. Explain the mechanism in terms of the merit order. Why did the *average price* fall less than you might expect, given how much solar was added?

*Your answer here.*

**6. A shortage: load shedding and scarcity prices**

Build the system **without imports** and with the coal unit out for maintenance. It becomes infeasible: in some hours demand exceeds everything available. Real systems then disconnect some load. Model this with `add_load_shedding`: a generator `load_shedding` at the bus with capacity equal to the peak load and a marginal cost equal to the **value of lost load** (VOLL, €/MWh). Then optimise again.

In [ ]:
def add_load_shedding(n: pypsa.Network, voll_EUR_per_MWh: float = 3000) -> None:
    """Add a load-shedding 'generator' priced at the value of lost load."""
    ...  # YOUR CODE HERE

In [ ]:
short = build_network(profiles, with_imports=False)
short.generators.loc["coal", "p_nom"] = 0
status = short.optimize(solver_name="highs", log_to_console=False)
assert status[1] != "optimal", "without imports and coal this system cannot meet demand"

add_load_shedding(short, 3000)
status = short.optimize(solver_name="highs", log_to_console=False)
assert status[1] == "optimal"

firm = (short.get_switchable_as_dense("Generator", "p_max_pu") * short.generators.p_nom).drop(columns="load_shedding")
gap = (short.loads_t.p_set["demand"] - firm.sum(axis=1)).clip(lower=0)
shed = short.generators_t.p["load_shedding"]
price = short.buses_t.marginal_price["HR"]
assert np.allclose(shed, gap, atol=1e-6), "shed exactly the shortfall, nothing more"
assert ((price == 3000) == (gap > 1e-6)).all(), "price = VOLL exactly in the shortage hours"
print(f"{(gap > 1e-6).sum()} shortage hours, {shed.sum():,.0f} MWh shed, max {shed.max():.0f} MW")

### Challenge

**7. Real Croatian load**

Replace the synthetic load with a real ENTSO-E week for Croatia (same dates, 13–19 January 2025, converted to naive UTC hours). Keep wind and solar synthetic for now. Write `profiles_with_real_load`, then optimise and compare cost, average price and imports with the synthetic week.

In [ ]:
def profiles_with_real_load(profiles: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Copy of `profiles` with load_MW replaced by ENTSO-E actual load for Croatia (hourly, naive UTC)."""
    ...  # YOUR CODE HERE

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
key = os.getenv("ENTSOE_API_KEY")

if key:
    real_prof = profiles_with_real_load(profiles, key)
    real_n = build_network(real_prof)
    real_n.optimize(solver_name="highs", log_to_console=False)
    summary = pd.DataFrame({
        "synthetic": [n.objective, n.buses_t.marginal_price["HR"].mean(), n.generators_t.p["imports"].sum()],
        "real load": [real_n.objective, real_n.buses_t.marginal_price["HR"].mean(), real_n.generators_t.p["imports"].sum()],
    }, index=["total cost EUR", "avg price EUR/MWh", "imports MWh"])
    print(summary.round(0))
else:
    print("Skipped: add ENTSOE_API_KEY to .env to run task 7.")

**8. What does a price of 3,000 €/MWh mean?**

In task 6 the price jumped to the VOLL in the shortage hours. What does that price represent economically, why is it useful for a model to produce it, and what would happen to your conclusions if you had picked VOLL = 10,000 €/MWh instead?

*Your answer here.*